In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from src.features import make_features

FEATS = ["hour", "weekday", "lag24_n", "lag168_n", "temp"]
UNSEEN_SITES = ["Panther", "Swan"]
MAX_MISSING = 0.20          # the same quality rule as notebook 01

df_clean = pd.read_parquet("../data/processed/electricity_clean.parquet")
temp = pd.read_parquet("../data/processed/air_temperature.parquet")
train_buildings = pd.read_csv("../configs/train_buildings.csv")["building"].tolist()
print("df_clean:", df_clean.shape, "index type:", type(df_clean.index).__name__)

# Buildings of the unseen sites: same rule (at most 20% missing), checked on 2017 only
candidates = [b for b in df_clean.columns if b.split("_")[0] in UNSEEN_SITES]
missing_2017 = df_clean.loc[df_clean.index.year == 2017, candidates].isna().mean()
unseen_buildings = missing_2017[missing_2017 <= MAX_MISSING].index.tolist()
print(len(candidates), "buildings on Panther/Swan,", len(unseen_buildings), "pass the 20% rule in 2017")
print(pd.Series([b.split("_")[0] for b in unseen_buildings]).value_counts().to_string())


def build(buildings):
    parts = []
    for b in buildings:
        f = make_features(b, df_clean, temp)
        num = f.columns.drop("building")
        f[num] = f[num].astype("float32")
        parts.append(f)
    out = pd.concat(parts)
    out["building"] = out["building"].astype("category")
    return out


data = build(train_buildings)                      # the 1,083 training buildings, 2016 + 2017
unseen = build(unseen_buildings)
unseen = unseen[unseen.index.year == 2017]         # unseen sites: 2017 only, never trained on
print(len(data), "rows for known buildings,", len(unseen), "rows for unseen buildings")
print("temperature range:", float(data["temp"].min()), "to", float(data["temp"].max()), "(should be °C)")

df_clean: (17544, 1578) index type: DatetimeIndex
124 buildings on Panther/Swan, 124 pass the 20% rule in 2017
Panther    105
Swan        19
19000152 rows for known buildings, 1086240 rows for unseen buildings
temperature range: -28.899999618530273 to 48.29999923706055 (should be °C)
